# Laboratório de Ciência de Dados
## Classificação supervisionada aplicada: modelos, limiares de decisão e custo dos erros


### Situação

Você integra a equipe de Ciência de Dados de uma fintech fictícia chamada **VGAbank**.

A empresa utiliza modelos de classificação para identificar transações potencialmente fraudulentas antes de sua aprovação.

O problema é que as fraudes são raras. Apenas cerca de **2% das transações** são fraudulentas.

Isso cria uma situação importante:

> Um modelo pode apresentar acurácia muito alta e, ainda assim, ser ruim para detectar fraudes.

Nesta aula, você não vai apenas "treinar classificadores". Você deverá ajudar a VGAbank a responder a uma pergunta de decisão:

> **Qual modelo e qual limiar de decisão devem ser utilizados para reduzir o custo esperado dos erros, garantindo que pelo menos 85% das fraudes sejam detectadas?**

### Regras de negócio

A VGAbank estima os seguintes custos médios:

- **Falso negativo (fraude não detectada): R$ 1.000**
- **Falso positivo (transação legítima enviada para revisão): R$ 15**

O modelo escolhido deve satisfazer:

> **Recall da classe fraude ≥ 85%**

Entre as configurações que satisfizerem essa exigência, deverá ser escolhida aquela com **menor custo total estimado**.

### O que você vai praticar

Nesta atividade você irá:

- lidar com um problema de classificação desbalanceada;
- comparar diferentes classificadores;
- observar por que a acurácia pode ser enganosa;
- trabalhar com probabilidades de classificação;
- alterar o **threshold** de decisão;
- calcular o custo de falsos positivos e falsos negativos;
- escolher um modelo com base em um critério de negócio.

> O foco desta aula não é revisar detalhadamente KNN, Árvores, Random Forest ou métricas. Esses conteúdos já foram estudados anteriormente. Aqui, eles serão utilizados para resolver um problema de decisão.

# 1. Preparando o ambiente

Utilizaremos `pandas`, `numpy`, `matplotlib` e componentes do `scikit-learn`.

Execute a célula abaixo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

RANDOM_STATE = 42

# 2. Construindo o conjunto de dados

Para que todos trabalhem com exatamente o mesmo conjunto de dados, utilizaremos uma base **sintética e reprodutível**, criada para simular um problema de detecção de fraude.

A base terá:

- **20.000 transações**;
- aproximadamente **2% de fraudes**;
- 12 atributos numéricos;
- uma variável-alvo chamada `fraude`.

Os atributos representam sinais numéricos que poderiam ser produzidos por sistemas antifraude, como comportamento de compra, características da transação e padrões históricos.

Execute a célula.

In [ ]:
X_array, y = make_classification(
    n_samples=20000,
    n_features=12,
    n_informative=7,
    n_redundant=3,
    n_repeated=0,
    n_classes=2,
    weights=[0.98, 0.02],
    class_sep=1.15,
    flip_y=0.004,
    random_state=RANDOM_STATE
)

nomes_atributos = [
    "valor_padronizado",
    "distancia_habitual",
    "hora_atipica",
    "tentativas_recentes",
    "risco_dispositivo",
    "risco_localizacao",
    "frequencia_24h",
    "variacao_valor",
    "idade_conta",
    "historico_chargeback",
    "divergencia_perfil",
    "score_comportamental"
]

X = pd.DataFrame(X_array, columns=nomes_atributos)
y = pd.Series(y, name="fraude")

dados = X.copy()
dados["fraude"] = y

dados.head()

## 2.1 Verificando o desbalanceamento

Antes de treinar qualquer modelo, observe quantas transações pertencem a cada classe.

In [ ]:
contagem = dados["fraude"].value_counts().sort_index()
percentual = dados["fraude"].value_counts(normalize=True).sort_index() * 100

distribuicao = pd.DataFrame({
    "quantidade": contagem,
    "percentual": percentual
})

distribuicao.index = ["Legítima (0)", "Fraude (1)"]
distribuicao

### Pergunta 1

Se um sistema simplesmente classificasse **todas** as transações como legítimas, aproximadamente qual seria sua acurácia?

Não calcule ainda. Faça uma estimativa observando a distribuição das classes.

# 3. O modelo ingênuo: a armadilha da acurácia

Vamos construir um "modelo" que nunca detecta fraude.

Ele sempre responde:

`0 = transação legítima`

Esse modelo não aprende nada, mas servirá como referência.

In [ ]:
y_pred_ingenuo = np.zeros(len(y), dtype=int)

acuracia_ingenua = accuracy_score(y, y_pred_ingenuo)
recall_ingenuo = recall_score(y, y_pred_ingenuo, zero_division=0)

print(f"Acurácia do modelo ingênuo: {acuracia_ingenua:.4f}")
print(f"Recall para fraudes: {recall_ingenuo:.4f}")

### Interpretação

O resultado mostra por que problemas desbalanceados exigem cuidado.

Um modelo pode alcançar uma acurácia próxima de 98% simplesmente porque quase todas as transações são legítimas.

Mas esse modelo detecta:

**0% das fraudes.**

Portanto, para a VGAbank, a pergunta não pode ser apenas:

> "Qual modelo tem maior acurácia?"

Precisamos observar também **quais erros são cometidos** e **quanto esses erros custam**.

# 4. Separando treino e teste

Vamos dividir os dados em:

- **70% para treinamento**
- **30% para teste**

Utilizaremos `stratify=y` para preservar aproximadamente a mesma proporção de fraudes nas duas partes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nFraudes no treino:", y_train.sum())
print("Fraudes no teste:", y_test.sum())

# 5. Três modelos candidatos

A equipe da VGAbank decidiu comparar três classificadores:

1. **Regressão Logística**
2. **KNN**
3. **Random Forest**

A padronização será aplicada automaticamente aos modelos que dela se beneficiam.

Não vamos realizar uma busca extensa de hiperparâmetros nesta aula. O objetivo é comparar modelos e, principalmente, estudar o efeito do **limiar de decisão**.

In [ ]:
modelos = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=7))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

# 6. Treinando os classificadores

Todos os modelos serão treinados utilizando exatamente os mesmos dados.

Execute a célula abaixo.

In [ ]:
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(f"{nome}: treinado")

# 7. Comparação inicial usando o threshold padrão

Por padrão, classificadores binários que produzem probabilidades costumam utilizar:

> **threshold = 0,50**

Isso significa que uma transação é classificada como fraude quando a probabilidade estimada de fraude é igual ou superior a 50%.

Vamos começar avaliando os três modelos com esse limiar.

In [ ]:
resultados_padrao = []

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)

    resultados_padrao.append({
        "modelo": nome,
        "acuracia": accuracy_score(y_test, y_pred),
        "precisao": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0)
    })

comparacao_padrao = pd.DataFrame(resultados_padrao)
comparacao_padrao.round(4)

### Pergunta 2

Observe a tabela.

**Qual modelo apresenta maior acurácia?**

**Esse mesmo modelo apresenta o maior recall para fraudes?**

Anote sua resposta antes de continuar.

# 8. Matriz de confusão e tipos de erro

Para a VGAbank, dois erros têm consequências muito diferentes:

### Falso positivo (FP)

Uma transação legítima é marcada como fraude.

Consequência: revisão manual ou bloqueio temporário.

**Custo estimado: R$ 15**

### Falso negativo (FN)

Uma fraude é classificada como legítima.

Consequência: a fraude passa pelo sistema.

**Custo estimado: R$ 1.000**

Vamos extrair esses valores para cada modelo.

In [ ]:
CUSTO_FP = 15
CUSTO_FN = 1000

def avaliar_predicoes(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()

    custo = fp * CUSTO_FP + fn * CUSTO_FN

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "acuracia": accuracy_score(y_real, y_pred),
        "precisao": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "f1": f1_score(y_real, y_pred, zero_division=0),
        "custo_total": custo
    }

In [ ]:
avaliacoes = []

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    resultado = avaliar_predicoes(y_test, y_pred)
    resultado["modelo"] = nome
    avaliacoes.append(resultado)

tabela_custos = pd.DataFrame(avaliacoes)

colunas = [
    "modelo", "TP", "FN", "FP", "TN",
    "acuracia", "precisao", "recall", "f1", "custo_total"
]

tabela_custos[colunas].round(4)

### Pergunta 3

Considere agora o custo dos erros.

**O modelo com maior acurácia é necessariamente o modelo de menor custo?**

**Qual tipo de erro tende a ser mais prejudicial neste problema: falso positivo ou falso negativo? Por quê?**

# 9. Probabilidade não é a mesma coisa que classe

Até agora utilizamos `predict()`, que retorna diretamente a classe prevista:

- `0` = legítima
- `1` = fraude

Mas muitos classificadores também conseguem retornar uma **probabilidade estimada**.

Usaremos `predict_proba()`.

Vamos observar as primeiras transações do conjunto de teste.

In [ ]:
modelo_exemplo = modelos["Regressão Logística"]

prob_fraude = modelo_exemplo.predict_proba(X_test)[:, 1]

pd.DataFrame({
    "probabilidade_fraude": prob_fraude[:10],
    "classe_real": y_test.iloc[:10].values
})

Observe que a probabilidade pode assumir qualquer valor entre 0 e 1.

Por exemplo:

- 0,03 → 3% de probabilidade estimada de fraude
- 0,47 → 47%
- 0,81 → 81%

Para transformar uma probabilidade em uma classe, precisamos definir um **limiar de decisão (threshold)**.

# 10. Alterando o threshold

Vamos criar manualmente as classificações da Regressão Logística.

Primeiro utilizaremos:

`threshold = 0.50`

In [ ]:
threshold = 0.50

y_pred_050 = (prob_fraude >= threshold).astype(int)

avaliacao_050 = avaliar_predicoes(y_test, y_pred_050)
pd.Series(avaliacao_050)

Agora vamos reduzir o limiar para:

`threshold = 0.30`

Isso significa que uma transação será marcada como fraude quando a probabilidade estimada for de pelo menos 30%.

In [ ]:
threshold = 0.30

y_pred_030 = (prob_fraude >= threshold).astype(int)

avaliacao_030 = avaliar_predicoes(y_test, y_pred_030)
pd.Series(avaliacao_030)

Agora teste:

`threshold = 0.20`

In [ ]:
threshold = 0.20

y_pred_020 = (prob_fraude >= threshold).astype(int)

avaliacao_020 = avaliar_predicoes(y_test, y_pred_020)
pd.Series(avaliacao_020)

### Pergunta 4

Compare os thresholds 0,50; 0,30 e 0,20.

Ao diminuir o threshold:

**a)** o número de fraudes detectadas tende a aumentar ou diminuir?

**b)** o número de falsos positivos tende a aumentar ou diminuir?

**c)** o custo total necessariamente diminui?

Não responda apenas "sim" ou "não". Observe os valores obtidos.

# 11. Comparando vários thresholds automaticamente

Testar limiares manualmente funciona para compreender o conceito, mas seria pouco eficiente testar dezenas de valores dessa maneira.

Vamos criar uma função que avalia vários thresholds.

In [ ]:
def avaliar_thresholds(nome_modelo, modelo, X_teste, y_teste, thresholds):
    probabilidades = modelo.predict_proba(X_teste)[:, 1]
    resultados = []

    for threshold in thresholds:
        y_pred = (probabilidades >= threshold).astype(int)
        avaliacao = avaliar_predicoes(y_teste, y_pred)

        resultados.append({
            "modelo": nome_modelo,
            "threshold": threshold,
            **avaliacao
        })

    return pd.DataFrame(resultados)

Vamos testar thresholds de **0,05 até 0,95**, em intervalos de 0,05.

In [ ]:
thresholds = np.arange(0.05, 1.00, 0.05)

resultado_lr = avaliar_thresholds(
    "Regressão Logística",
    modelos["Regressão Logística"],
    X_test,
    y_test,
    thresholds
)

resultado_lr[[
    "threshold", "recall", "precisao", "FP", "FN", "custo_total"
]].round(4)

# 12. Visualizando o efeito do threshold

Vamos observar como o threshold altera o recall.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(resultado_lr["threshold"], resultado_lr["recall"], marker="o")
plt.axhline(0.85, linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("Recall")
plt.title("Regressão Logística — Recall por threshold")
plt.grid(True, alpha=0.3)
plt.show()

A linha horizontal representa a exigência da empresa:

> **Recall mínimo = 0,85**

Agora observe o custo total.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(resultado_lr["threshold"], resultado_lr["custo_total"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Custo total estimado (R$)")
plt.title("Regressão Logística — Custo por threshold")
plt.grid(True, alpha=0.3)
plt.show()

# 13. Regra de decisão da empresa

A VGAbank estabeleceu a seguinte regra:

1. O modelo deve detectar **pelo menos 85% das fraudes**.
2. Entre as configurações que satisfizerem essa condição, deve ser escolhida aquela com **menor custo total**.

Vamos aplicar a regra à Regressão Logística.

In [ ]:
candidatos_lr = resultado_lr[
    resultado_lr["recall"] >= 0.85
].copy()

melhor_lr = candidatos_lr.sort_values("custo_total").head(1)

melhor_lr[[
    "modelo", "threshold", "recall", "precisao",
    "FP", "FN", "custo_total"
]].round(4)

### Pergunta 5

Qual foi o melhor threshold encontrado para a Regressão Logística considerando simultaneamente:

- recall ≥ 85%;
- menor custo?

Anote o valor.

# 14. Repetindo o processo para todos os modelos

Agora vamos aplicar exatamente o mesmo procedimento aos três classificadores.

Observe que o critério de negócio será o mesmo para todos.

In [ ]:
resultados_todos = []

for nome, modelo in modelos.items():
    resultado = avaliar_thresholds(
        nome,
        modelo,
        X_test,
        y_test,
        thresholds
    )
    resultados_todos.append(resultado)

resultados_thresholds = pd.concat(resultados_todos, ignore_index=True)

resultados_thresholds.head()

Vamos selecionar somente as configurações que atendem à exigência mínima de recall.

In [ ]:
candidatos = resultados_thresholds[
    resultados_thresholds["recall"] >= 0.85
].copy()

print("Número de configurações que atendem ao requisito:", len(candidatos))

candidatos[[
    "modelo", "threshold", "recall", "precisao",
    "FP", "FN", "custo_total"
]].sort_values("custo_total").head(15).round(4)

# 15. Melhor configuração de cada modelo

Agora vamos identificar, para cada algoritmo, qual threshold produz o menor custo entre aqueles que atendem ao requisito de recall.

In [ ]:
melhor_por_modelo = (
    candidatos
    .sort_values("custo_total")
    .groupby("modelo", as_index=False)
    .first()
)

melhor_por_modelo[[
    "modelo", "threshold", "recall", "precisao",
    "FP", "FN", "custo_total"
]].round(4)

# 16. Decisão final

Finalmente, vamos selecionar a configuração de menor custo entre todos os modelos.

In [ ]:
melhor_global = candidatos.sort_values("custo_total").head(1)

melhor_global[[
    "modelo", "threshold", "recall", "precisao",
    "FP", "FN", "custo_total"
]].round(4)

## Sua recomendação à VGAbank

Com base nos resultados, complete:

**Modelo recomendado:** __________________________________

**Threshold recomendado:** _______________________________

**Recall obtido:** ______________________________________

**Falsos negativos:** ___________________________________

**Falsos positivos:** ___________________________________

**Custo total estimado:** ________________________________

Agora escreva uma justificativa de **3 a 5 linhas** explicando por que essa configuração deve ser escolhida.

Sua justificativa deve mencionar obrigatoriamente:

- a exigência de recall mínimo;
- o custo dos erros;
- por que a acurácia isoladamente não é suficiente.

# 17. Desafio final — mudança no custo do negócio

A diretoria informou que o impacto financeiro de uma fraude aumentou.

Agora:

- **Falso negativo: R$ 2.000**
- **Falso positivo: R$ 15**

A exigência de recall continua sendo:

> **recall ≥ 85%**

Sua tarefa é verificar se a recomendação muda.

Não é necessário treinar novamente os modelos. As probabilidades previstas continuam as mesmas.

Altere apenas o custo do falso negativo e refaça a avaliação.

In [ ]:
# NOVO CENÁRIO

NOVO_CUSTO_FP = 15
NOVO_CUSTO_FN = 2000

def avaliar_predicoes_novo_custo(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()

    custo = fp * NOVO_CUSTO_FP + fn * NOVO_CUSTO_FN

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "acuracia": accuracy_score(y_real, y_pred),
        "precisao": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "f1": f1_score(y_real, y_pred, zero_division=0),
        "custo_total": custo
    }

In [ ]:
def avaliar_thresholds_novo_custo(nome_modelo, modelo, X_teste, y_teste, thresholds):
    probabilidades = modelo.predict_proba(X_teste)[:, 1]
    resultados = []

    for threshold in thresholds:
        y_pred = (probabilidades >= threshold).astype(int)
        avaliacao = avaliar_predicoes_novo_custo(y_teste, y_pred)

        resultados.append({
            "modelo": nome_modelo,
            "threshold": threshold,
            **avaliacao
        })

    return pd.DataFrame(resultados)

resultados_novo_custo = []

for nome, modelo in modelos.items():
    resultado = avaliar_thresholds_novo_custo(
        nome,
        modelo,
        X_test,
        y_test,
        thresholds
    )
    resultados_novo_custo.append(resultado)

resultado_cenario_2 = pd.concat(resultados_novo_custo, ignore_index=True)

candidatos_cenario_2 = resultado_cenario_2[
    resultado_cenario_2["recall"] >= 0.85
].copy()

melhor_cenario_2 = candidatos_cenario_2.sort_values("custo_total").head(1)

melhor_cenario_2[[
    "modelo", "threshold", "recall", "precisao",
    "FP", "FN", "custo_total"
]].round(4)

### Perguntas do desafio

**1. O modelo recomendado permaneceu o mesmo?**

**2. O threshold recomendado permaneceu o mesmo?**

**3. Por que alterar o custo do falso negativo pode modificar a decisão mesmo sem treinar novamente os modelos?**

**4. O que esse experimento mostra sobre a relação entre desempenho estatístico e decisão de negócio?**

# 18. Conclusão

Nesta aula, você partiu de um problema de classificação aparentemente simples:

> fraude ou não fraude?

Mas percebeu que a escolha do modelo depende de muito mais do que a acurácia.

O fluxo realizado foi:

**problema → dados → modelos → probabilidades → threshold → matriz de confusão → custo dos erros → restrição de recall → decisão**

A principal ideia da atividade é:

> **O melhor classificador não é necessariamente aquele que apresenta a maior acurácia. A escolha depende do objetivo do problema, do custo dos diferentes tipos de erro e do limiar utilizado para converter probabilidades em decisões.**

Em aplicações reais, métricas estatísticas são fundamentais, mas precisam ser interpretadas dentro do contexto em que o modelo será utilizado.

# 19. Reflexão final

Responda brevemente:

**1. Por que a acurácia pode ser enganosa em um conjunto muito desbalanceado?**

**2. Qual é a diferença entre `predict()` e `predict_proba()`?**

**3. O que é o threshold de classificação?**

**4. O que tende a acontecer com o recall quando diminuímos o threshold?**

**5. Por que falso positivo e falso negativo não devem necessariamente receber o mesmo peso?**

**6. Por que o modelo de menor custo pode não ser o de maior acurácia?**

**7. Em uma aplicação real, quem deveria definir os custos e restrições do problema: apenas o cientista de dados ou também especialistas do domínio e gestores? Explique.**